<a href="https://colab.research.google.com/github/vedheshatthi104ff-hub/Project/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
if not os.path.exists("/content/Internship"):
    !git clone https://github.com/vedheshatthi104ff-hub/Internship.git /content/Internship
%cd /content/Internship
!pip install -q duckdb huggingface_hub scikit-learn matplotlib

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
print("token loaded:", bool(HF_TOKEN))

Cloning into '/content/Internship'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 178 (delta 79), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 1.87 MiB | 18.57 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/Internship
token loaded: True


In [3]:
import duckdb, pandas as pd, numpy as np, os, json
from datetime import date, timedelta
from huggingface_hub import list_repo_files

REPO = "FlyRank/internship-warehouse"
files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

def month_files(months):
    fs = [f for f in files if "fact_content_daily_performance" in f and any(m in f for m in months)]
    print(months, "->", len(fs), "files")          # if 0, print(files[:20]) and fix the pattern
    return "[" + ",".join(f"'hf://datasets/{REPO}/{f}'" for f in fs) + "]"

def build(decision, months):
    D = date.fromisoformat(decision); start = D - timedelta(days=89); end = D + timedelta(days=30)
    FILES = month_files(months)
    q = f"""
    WITH f AS (
      SELECT client_hash_id c, content_hash_id p, report_date d,
             gsc_impressions imp, gsc_clicks clk, gsc_avg_position pos
      FROM read_parquet({FILES})
      WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
    SELECT c AS client_hash_id, p AS content_hash_id,
      SUM(imp) FILTER (WHERE d <= DATE '{D}' AND d > DATE '{D - timedelta(days=30)}') AS last30,
      SUM(imp) FILTER (WHERE d <= DATE '{D - timedelta(days=30)}')                    AS prior60,
      SUM(imp) FILTER (WHERE d <= DATE '{D}')                                          AS imp90,
      SUM(clk) FILTER (WHERE d <= DATE '{D}')                                          AS clicks90,
      SUM(pos*imp) FILTER (WHERE d <= DATE '{D}') / NULLIF(SUM(imp) FILTER (WHERE d <= DATE '{D}'),0) AS pos90,
      COUNT(*) FILTER (WHERE d <= DATE '{D}' AND imp > 0)                              AS active_days,
      STDDEV(imp) FILTER (WHERE d <= DATE '{D}')                                       AS imp_std,
      COALESCE(SUM(imp) FILTER (WHERE d > DATE '{D}'), 0)                              AS next30
    FROM f GROUP BY 1, 2
    HAVING SUM(imp) FILTER (WHERE d <= DATE '{D}' AND d > DATE '{D - timedelta(days=30)}') >= 100"""
    df = con.execute(q).df().fillna(0)
    df["ctr90"] = df["clicks90"] / df["imp90"].replace(0, np.nan)
    df["trend_ratio"] = df["last30"] / (df["prior60"] / 2 + 1)
    df["y"] = (df["next30"] < 0.8 * df["last30"]).astype(int)   # future observed outcome
    return df.fillna(0)

os.makedirs("work/outputs", exist_ok=True)
dev  = build("2026-03-31", ["2026-01", "2026-02", "2026-03", "2026-04"])   # develop here
dev.to_parquet("/content/dev.parquet")
print(dev.shape, "| label rate:", round(dev["y"].mean(), 3))

['2026-01', '2026-02', '2026-03', '2026-04'] -> 4 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100849, 13) | label rate: 0.498


In [4]:
print("clients:", dev["client_hash_id"].nunique(), "| pages:", dev["content_hash_id"].nunique())
print("share of pages with zero next30 impressions:", round((dev["next30"] == 0).mean(), 3))
print("pages per client (top 5):")
print(dev["client_hash_id"].value_counts().head(5))

clients: 44 | pages: 100849
share of pages with zero next30 impressions: 0.005
pages per client (top 5):
client_hash_id
client_73cda7b4e4f265ea    21531
client_62f4a7e64f5e0096    17192
client_23a62021009f63c4    12746
client_e547b89c05043229     8485
client_fef1a8f436438636     7977
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

FEATS = ["last30","prior60","imp90","clicks90","ctr90","pos90","active_days","imp_std","trend_ratio"]
K = 50

def baseline_score(d):   # transparent rule, no labels: demand x recent drop
    demand = np.log1p(d["last30"]).rank(pct=True)
    drop = (1 - d["trend_ratio"]).clip(0, 1)
    return 0.4 * demand + 0.6 * drop

def patk(y, s, k=K):
    return y.iloc[np.argsort(-np.asarray(s))[:k]].mean()

def report(name, y, s):
    return {"method": name, "auc": round(roc_auc_score(y, s), 3),
            "ap": round(average_precision_score(y, s), 3), "p_at_k": round(patk(y, s), 3)}

X, y, g = dev[FEATS], dev["y"], dev["client_hash_id"]
oof = {"logreg": np.zeros(len(dev)), "rf": np.zeros(len(dev))}
for tr, te in GroupKFold(5).split(X, y, g):               # whole clients held out
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X.iloc[tr], y.iloc[tr])
    rf = RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(X.iloc[tr], y.iloc[tr])
    oof["logreg"][te] = lr.predict_proba(X.iloc[te])[:, 1]
    oof["rf"][te]     = rf.predict_proba(X.iloc[te])[:, 1]

rows = [report("rule baseline", y, baseline_score(dev)),
        report("logistic regression", y, oof["logreg"]),
        report("random forest", y, oof["rf"])]
dev_results = pd.DataFrame(rows); print(dev_results)

                method    auc     ap  p_at_k
0        rule baseline  0.475  0.483    0.62
1  logistic regression  0.548  0.511    0.86
2        random forest  0.588  0.557    0.68


In [6]:
# 1) label rate per big client: does the base rate differ a lot by client?
top5 = dev["client_hash_id"].value_counts().head(5).index
print(dev[dev["client_hash_id"].isin(top5)].groupby("client_hash_id")["y"]
        .agg(rows="size", label_rate="mean").round(3), "\n")

# 2) per-fold results (whole clients held out in each fold)
res = []
for i, (tr, te) in enumerate(GroupKFold(5).split(X, y, g)):
    yt = y.iloc[te]
    top = np.argsort(-oof["rf"][te])[:K]
    res.append({"fold": i, "rows": len(te), "clients": g.iloc[te].nunique(),
                "base_rate": round(yt.mean(), 3),
                "p@K baseline": round(patk(yt, baseline_score(dev.iloc[te])), 3),
                "p@K logreg": round(patk(yt, oof["logreg"][te]), 3),
                "p@K rf": round(patk(yt, oof["rf"][te]), 3),
                "clients in rf top-K": g.iloc[te].iloc[top].nunique()})
print(pd.DataFrame(res))

# 3) per-client AUC for clients with enough rows and both classes
pc = []
for c, idx in dev.groupby("client_hash_id").indices.items():
    if len(idx) >= 500 and 0 < y.iloc[idx].mean() < 1:
        pc.append(roc_auc_score(y.iloc[idx], oof["rf"][idx]))
print("clients scored:", len(pc), "| median AUC:", round(np.median(pc), 3),
      "| share above 0.5:", round(np.mean(np.array(pc) > 0.5), 2))

                          rows  label_rate
client_hash_id                            
client_23a62021009f63c4  12746       0.447
client_62f4a7e64f5e0096  17192       0.709
client_73cda7b4e4f265ea  21531       0.407
client_e547b89c05043229   8485       0.298
client_fef1a8f436438636   7977       0.694 

   fold   rows  clients  base_rate  p@K baseline  p@K logreg  p@K rf  \
0     0  21531        1      0.407          0.50        0.82    0.58   
1     1  19819       11      0.719          0.70        0.94    0.80   
2     2  19818       12      0.520          0.76        0.70    0.66   
3     3  19819       14      0.351          0.46        0.66    0.64   
4     4  19862        6      0.500          0.26        0.28    0.72   

   clients in rf top-K  
0                    1  
1                    3  
2                    5  
3                    4  
4                    5  
clients scored: 20 | median AUC: 0.65 | share above 0.5: 1.0


In [7]:
# 1) does a recent drop predict MORE or FEWER future declines?
dev["trend_bin"] = pd.qcut(dev["trend_ratio"], 10, duplicates="drop")
print(dev.groupby("trend_bin", observed=True)["y"].agg(n="size", label_rate="mean").round(3), "\n")

# 2) precision and lift at several K (pooled out-of-fold)
base = y.mean(); rows = []
for k in [50, 200, 1000]:
    for name, s in [("baseline", baseline_score(dev)), ("logreg", oof["logreg"]), ("rf", oof["rf"])]:
        p = patk(y, s, k); rows.append({"K": k, "method": name, "p_at_k": round(p, 3), "lift": round(p / base, 2)})
print("base rate:", round(base, 3))
print(pd.DataFrame(rows).pivot(index="method", columns="K", values=["p_at_k", "lift"]))

                      n  label_rate
trend_bin                          
(0.00669, 0.786]  10085       0.542
(0.786, 0.999]    10085       0.468
(0.999, 1.225]    10085       0.532
(1.225, 1.501]    10085       0.548
(1.501, 1.912]    10085       0.559
(1.912, 2.688]    10084       0.564
(2.688, 4.544]    10085       0.543
(4.544, 12.101]   10085       0.510
(12.101, 275.0]   10095       0.422
(275.0, 83788.0]  10075       0.292 

base rate: 0.498
         p_at_k                lift            
K          50     200    1000  50    200   1000
method                                         
baseline   0.62  0.550  0.536  1.24  1.10  1.08
logreg     0.86  0.725  0.605  1.73  1.46  1.21
rf         0.68  0.650  0.611  1.37  1.31  1.23


In [8]:
def add_rel(d):
    d = d.copy(); grp = d.groupby("client_hash_id")
    d["trend_rel"]   = d["trend_ratio"] - grp["trend_ratio"].transform("median")
    d["last30_rank"] = grp["last30"].rank(pct=True)
    d["pos_rel"]     = d["pos90"] - grp["pos90"].transform("median")
    return d

dev2 = add_rel(dev)
FEATS2 = FEATS + ["trend_rel", "last30_rank", "pos_rel"]

def cv_oof(cols, kind):
    Xc = dev2[cols]; o = np.zeros(len(dev2))
    for tr, te in GroupKFold(5).split(Xc, y, g):
        m = (RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0) if kind == "rf"
             else make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
        o[te] = m.fit(Xc.iloc[tr], y.iloc[tr]).predict_proba(Xc.iloc[te])[:, 1]
    return o

print(pd.DataFrame([report("rf + client-relative", y, cv_oof(FEATS2, "rf")),
                    report("logreg + client-relative", y, cv_oof(FEATS2, "lr"))]))

                     method    auc     ap  p_at_k
0      rf + client-relative  0.557  0.527    0.72
1  logreg + client-relative  0.516  0.491    0.60


In [9]:
tmp = dev.copy(); tmp["leak"] = tmp["next30"] / (tmp["last30"] + 1)
Xl = tmp[FEATS + ["leak"]]; s = np.zeros(len(tmp))
for tr, te in GroupKFold(5).split(Xl, y, g):
    s[te] = RandomForestClassifier(100, n_jobs=-1, random_state=0).fit(Xl.iloc[tr], y.iloc[tr]).predict_proba(Xl.iloc[te])[:, 1]
print("with deliberate leak AUC:", round(roc_auc_score(y, s), 3), "(should be ~1.0)")
print("honest random forest AUC:", dev_results.loc[2, "auc"])

with deliberate leak AUC: 1.0 (should be ~1.0)
honest random forest AUC: 0.588


Leakage audit. (1) The feature window ends at the decision date; every feature sum uses dates up to and including it, and the label uses only later dates, so the windows don't overlap. (2) No product scores or flags are used. (3) Deliberate-leak test: adding a column built from the future window raised AUC from 0.588 to 1.0, so a leak of that kind would be caught. The leak column is not in any reported model. (4) Validation holds out whole clients, so no page appears in both train and test. (5) The final month (June) is built only after all choices are frozen and is scored once.

Frozen before the sealed test. Label: next-30-day impressions < 80% of last-30-day impressions, pages with ≥100 impressions in the last 30 days. Features: the 9 original features (no client-relative features, which lowered AP on dev). Primary metric: average precision, with lift at K=200 and K=1000 as secondary and P@50 shown only for continuity. Model for the queue: random forest; logistic regression and the week-4 rule baseline reported alongside. Baseline v1 is kept unchanged. The June test is run once and no changes follow it.

In [10]:
test = build("2026-05-31", ["2026-03", "2026-04", "2026-05", "2026-06"])   # label = June
Xt, yt = test[FEATS], test["y"]

rf_f = RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(X, y)
lr_f = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X, y)
scores = {"rule baseline": baseline_score(test).values,
          "logistic regression": lr_f.predict_proba(Xt)[:, 1],
          "random forest": rf_f.predict_proba(Xt)[:, 1]}
test["p_rf"] = scores["random forest"]

base_t = yt.mean(); rows = []
for name, s in scores.items():
    r = {"method": name, "auc": round(roc_auc_score(yt, s), 3), "ap": round(average_precision_score(yt, s), 3)}
    for k in [50, 200, 1000]:
        p = patk(yt, s, k); r[f"p@{k}"] = round(p, 3); r[f"lift@{k}"] = round(p / base_t, 2)
    rows.append(r)
test_results = pd.DataFrame(rows)
print("June base rate:", round(base_t, 3), "| rows:", len(test), "| clients:", test["client_hash_id"].nunique())
print(test_results)

# bootstrap range for AP (resampling pages; approximate, pages within a client are correlated)
rng = np.random.default_rng(0); ci = {}
for name, s in scores.items():
    vals = []
    for _ in range(100):
        i = rng.integers(0, len(yt), len(yt))
        vals.append(average_precision_score(yt.values[i], s[i]))
    ci[name] = (round(np.percentile(vals, 2.5), 3), round(np.percentile(vals, 97.5), 3))
print("AP 95% bootstrap range:", ci)

json.dump({"dev": dev_results.to_dict("records"), "sealed_test": test_results.to_dict("records"),
           "ap_bootstrap_range": {k: list(v) for k, v in ci.items()},
           "dev_rows": len(dev), "test_rows": len(test), "june_base_rate": round(float(base_t), 3)},
          open("work/outputs/capstone_results.json", "w"), indent=2)

['2026-03', '2026-04', '2026-05', '2026-06'] -> 4 files


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

June base rate: 0.643 | rows: 111247 | clients: 50
                method    auc     ap  p@50  lift@50  p@200  lift@200  p@1000  \
0        rule baseline  0.520  0.671  0.82     1.28  0.795      1.24   0.766   
1  logistic regression  0.596  0.702  0.72     1.12  0.645      1.00   0.700   
2        random forest  0.630  0.721  0.66     1.03  0.705      1.10   0.757   

   lift@1000  
0       1.19  
1       1.09  
2       1.18  
AP 95% bootstrap range: {'rule baseline': (np.float64(0.668), np.float64(0.676)), 'logistic regression': (np.float64(0.698), np.float64(0.705)), 'random forest': (np.float64(0.718), np.float64(0.724))}


In [11]:
rng = np.random.default_rng(1)
yv = yt.values
cl_idx = {c: np.where(test["client_hash_id"].values == c)[0] for c in test["client_hash_id"].unique()}
cl = list(cl_idx)
diffs = {"rf - baseline": [], "rf - logreg": []}
for _ in range(100):
    pick = rng.choice(cl, len(cl), replace=True)
    idx = np.concatenate([cl_idx[c] for c in pick])
    ap = {n: average_precision_score(yv[idx], s[idx]) for n, s in scores.items()}
    diffs["rf - baseline"].append(ap["random forest"] - ap["rule baseline"])
    diffs["rf - logreg"].append(ap["random forest"] - ap["logistic regression"])
for k, v in diffs.items():
    print(k, "| mean:", round(np.mean(v), 3), "| 95% range:", np.round(np.percentile(v, [2.5, 97.5]), 3))

rf - baseline | mean: 0.051 | 95% range: [0.015 0.092]
rf - logreg | mean: 0.023 | 95% range: [-0.017  0.062]


In [12]:
import matplotlib.pyplot as plt
os.makedirs("charts", exist_ok=True)

ax = test_results.set_index("method")["ap"].plot.bar(rot=0, figsize=(6, 4))
ax.axhline(base_t, color="gray", linestyle="--", label=f"base rate {base_t:.3f} (chance)")
ax.set_ylim(0.55, 0.78); ax.set_ylabel("Average precision (sealed June test)")
ax.set_title("Average precision vs chance"); ax.legend()
plt.tight_layout(); plt.savefig("charts/precision_at_k.png", dpi=150); plt.close()

imp = pd.Series(rf_f.feature_importances_, index=FEATS).sort_values()
imp.plot.barh(figsize=(6, 4), title="Random forest feature importance")
plt.tight_layout(); plt.savefig("charts/feature_importance.png", dpi=150); plt.close()

q = test.sort_values("p_rf", ascending=False).head(50).copy()
q["reason_code"] = np.where(q["trend_ratio"] < 0.8, "declining_with_demand", "high_model_risk")
q["action"] = np.where(q["reason_code"] == "declining_with_demand", "review_for_refresh", "monitor")
q[["content_hash_id", "p_rf", "reason_code", "action", "last30", "trend_ratio"]].head(10)

,content_hash_id,p_rf,reason_code,action,last30,trend_ratio
46705,content_edd97d4af66f291d,0.969384,high_model_risk,monitor,202.0,33.666667
87107,content_444850d777e4f2fd,0.954751,high_model_risk,monitor,291.0,12.125000
67618,content_fbb2d2cfb20b93b8,0.953613,high_model_risk,monitor,265.0,20.384615
104579,content_11ec8b3729f821e0,0.949955,high_model_risk,monitor,409.0,3.338776
106719,content_7fd526db8a9cdb83,0.943958,high_model_risk,monitor,192.0,54.857143
31916,content_4319262b9c05b6cc,0.943167,high_model_risk,monitor,268.0,89.333333
95847,content_f319316f4e46d472,0.943046,high_model_risk,monitor,209.0,46.444444
93133,content_0e49862d87b5068a,0.934210,high_model_risk,monitor,121.0,80.666667
64851,content_1727399338b93070,0.930004,high_model_risk,monitor,153.0,18.000000
91021,content_ebec9ec80d64d6e5,0.928302,high_model_risk,monitor,828.0,5.341935
